In [1]:
#This is a bunch of initialization
import numpy as np
import matplotlib.pyplot as plt 
import gym
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset
env = gym.make('CartPole-v0')
print(env.observation_space)
print(env.action_space)   # 0: left, 1: right

env.reset()
observation_new = None
for i in range (200):
    env.render()
    observation_new, reward, done, info = env.step(1)
    print(i, observation_new, reward, done) 
    if done:
        env.reset()
env.close()

Box(-3.4028234663852886e+38, 3.4028234663852886e+38, (4,), float32)
Discrete(2)
0 [ 0.01629223  0.24148882 -0.02415082 -0.26293981] 1.0 False
1 [ 0.02112201  0.43694703 -0.02940962 -0.56314127] 1.0 False
2 [ 0.02986095  0.63246905 -0.04067245 -0.86494265] 1.0 False
3 [ 0.04251033  0.82812031 -0.0579713  -1.17013124] 1.0 False
4 [ 0.05907273  1.0239463  -0.08137392 -1.4804108 ] 1.0 False
5 [ 0.07955166  1.21996156 -0.11098214 -1.79735831] 1.0 False
6 [ 0.10395089  1.41613698 -0.14692931 -2.12237288] 1.0 False
7 [ 0.13227363  1.61238477 -0.18937676 -2.45661475] 1.0 False
8 [ 0.16452133  1.80854121 -0.23850906 -2.80093358] 1.0 True
9 [-0.01252829  0.2052544  -0.02546578 -0.32784059] 1.0 False
10 [-0.0084232   0.40072946 -0.03202259 -0.62844425] 1.0 False
11 [-4.08610762e-04  5.96283348e-01 -4.45914753e-02 -9.31037720e-01] 1.0 False
12 [ 0.01151706  0.79197781 -0.06321223 -1.23739317] 1.0 False
13 [ 0.02735661  0.98785235 -0.08796009 -1.54919019] 1.0 False
14 [ 0.04711366  1.18391288 -0.11

132 [-0.01191575  0.75458382  0.00484724 -1.08400511] 1.0 False
133 [ 0.00317592  0.94964148 -0.01683286 -1.37516308] 1.0 False
134 [ 0.02216875  1.14496969 -0.04433613 -1.67306251] 1.0 False
135 [ 0.04506815  1.3405775  -0.07779738 -1.97921645] 1.0 False
136 [ 0.0718797   1.5364271  -0.1173817  -2.29495164] 1.0 False
137 [ 0.10260824  1.7324186  -0.16328074 -2.6213469 ] 1.0 False
138 [ 0.13725661  1.92837221 -0.21570768 -2.95915934] 1.0 True
139 [ 0.00099402  0.20691081  0.02397963 -0.24325881] 1.0 False
140 [ 0.00513224  0.40168218  0.01911445 -0.52828254] 1.0 False
141 [ 0.01316588  0.59653006  0.0085488  -0.81488167] 1.0 False
142 [ 0.02509649  0.79153392 -0.00774883 -1.10486344] 1.0 False
143 [ 0.04092716  0.98675691 -0.0298461  -1.39996727] 1.0 False
144 [ 0.0606623   1.18223684 -0.05784545 -1.70183003] 1.0 False
145 [ 0.08430704  1.3779753  -0.09188205 -2.01194372] 1.0 False
146 [ 0.11186655  1.57392452 -0.13212092 -2.33160305] 1.0 False
147 [ 0.14334504  1.76997165 -0.17875298 

In [2]:
class Q_net(nn.Module):
    def __init__(self):
        super(Q_net, self).__init__()
        self.fc1 = nn.Linear(4,16)
        self.ReLU1 = nn.LeakyReLU(inplace=True)
        self.fc2 = nn.Linear(16,24)
        self.ReLU2 = nn.LeakyReLU(inplace=True)
        self.fc3 = nn.Linear(24,2)
    def forward(self, x):
        out = self.ReLU1(self.fc1(x))
        out = self.ReLU2(self.fc2(out))
        out = self.fc3(out)
        return out

In [3]:
#Create data set (same as mountain car project)
class RLDataset(Dataset):
    def __init__(self, samples, transform = None, target_transform = None):
        #samples = [(s,a,r,s_), ...]
        self.samples = self.transform(samples)
    def __getitem__(self, index):
        #if self.transform is not None:
        #    img = self.transform(img) 
        return self.samples[index]
    def __len__(self):
        return len(self.samples)
    def transform(self, samples):
        transSamples = []
        for (s,a,r,s_) in samples:
            sT = torch.tensor(s,).float()
            sT_ = torch.tensor(s_).float()
            transSamples.append((sT, a, r, sT_))
        return transSamples

In [4]:
# get samples: (s,a,r,s') for many episodes from env, following e-greedy
# arguments: 
#    - env: environment
#    - model: current policy
#    - episodes: the number of episodes to be generated
#    - max_steps: the max length of one episode
#    - epsilon: epsilon in e-greedy
# return: train_samples[], a list of (s,a,r,s') for (episodes) episodes
# Note: the reward is modified: reward =-1.0 if done.
#       
def GetSamplesFromEnv(env, model, episodes, max_steps=200, epsilon = 0.8):
    train_samples = []
    each_sample = None
    env.reset()
    observation_new = None   # state s
    observation_old = None   # state s'
    model.eval()
    for i_episode in range(episodes):
        #observation_new = env.reset()
        observation_old = env.reset()
        for t in range(max_steps):
            #env.render()
            #print(observation)
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                inputT = torch.tensor(observation_old).float()
                action = torch.argmax(model(inputT)).item()
                #print(action)
            observation_new, reward, done, info = env.step(action)
            if done:
                reward = -1.0
            #print(reward)
            if t > 0 :
                each_sample = (observation_old, action, reward, observation_new)
                train_samples.append(each_sample)

            observation_old = observation_new

            if done:
            	#Failed samples are not printed
                #if t >= 195:
                #         print("Episode finished after {} timesteps".format(t+1))
                break
    print("avg length is {} timesteps".format(len(train_samples)/episodes))
    return train_samples


In [5]:
def GetSamplesFromEnv(env, model, episodes, max_steps, epsilon = 0.8):
    train_samples = []
    each_sample = None
    env.reset()
    observation_new = None
    observation_old = None
    model.eval()
    for i_episode in range(episodes):
        #observation_new = env.reset()
        observation_old = env.reset()
        for t in range(max_steps):
            #env.render()
            #print(observation)
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                #inputT = torch.tensor(observation_new).float()
                inputT = torch.tensor(observation_old).float()
                action = torch.argmax(model(inputT)).item()
                #print(action)
            observation_new, reward, done, info = env.step(action)
            if done:
                reward = -1.0
            #print(reward)
            if t > 0 :
                each_sample = (observation_old, action, reward, observation_new)
                train_samples.append(each_sample)

            observation_old = observation_new

            if done:
            	#Failed samples are not printed
                if t >= 195:
                         print("Episode finished after {} timesteps".format(t+1))
                break
    return train_samples

In [6]:

# train the q_net, based on a fixed dataset.
# arguments
#     - target_net:  the target network, copied from q_net every 100 iterations
#     - q_net:  the Q network, updated every iteration
#     - trainloader: load the dataset for training
#     - criterion: loss function, e.g. nn.MSELoss()
#     - optimizer: optimizer, e.g., Adam
#     - device: cpu or gpu
#     - epoch_total: the number of epochs for training
#     - gamma: reward discount
def TrainNet(target_net, q_net, trainloader, criterion, optimizer, device, epoch_total, gamma):
    running_loss = 0.0
    iter_times = 0
    target_net.eval()
    q_net.train()
    for epoch in range(epoch_total):
        running_loss = 0.0
#        if epoch == epoch_total: 
#            break        
        for i, data in enumerate(trainloader, 0):
            if iter_times % 100 == 0:
                target_net.load_state_dict(q_net.state_dict())
            s,a,r,s_ = data
            optimizer.zero_grad()

            #output = Q_predicted.
            q_t0 = q_net(s)   # predict of q_net
            q_t1 = target_net(s_).detach()
            target = r +gamma * (torch.max(q_t1,dim=1)[0])
            
            loss = criterion(target, torch.gather(q_t0, dim=1, index=a.unsqueeze(1)).squeeze(1)) 
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            iter_times += 1
    target_net.load_state_dict(q_net.state_dict())    
    #print('Finished Training')



In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
q_net = Q_net()     # initial q_net
target_net = Q_net()  # initial target_net differently

criterion = nn.MSELoss()  # loss function
optimizer = torch.optim.Adam(q_net.parameters(),lr=0.01)   # optimmizer

train_samples = []    # define a list of (s,a,r,s')
num_datasets = 300       # number of datasets

epsilon = 1.             # initial epsilon
END_eps= num_datasets//2  # epsilon decays in the first half
START_eps=1
eps_step = epsilon / (END_eps-START_eps) # decay amount after each dataset

episodes =10     # for each dataset, update 10 new episodes
shortest_len = episodes *195  # initial the best length for 10 episodes (1600)

epochs = 12  # train for 12 epochs at each dataset

In [8]:
# training loop
# for each new dataset:
#     1) set a new epsilon
#     2) collect episodes(e.g.10) episodes by GetSamplesFromEnv()
#     3) update the dataset train_samples (limited to 4000)
#     4) trainloader the dataset
#     5) train q_net on the updated dataset

for i in range(num_datasets):
    # monitor epsilon for each dataset
    print('dataset:', i, 'epsilon', epsilon)
    # decay epsilon
    if END_eps-2 >= i >= START_eps:
        epsilon -= eps_step
    # collect new 10 episodes 
    tmpSample = GetSamplesFromEnv(env,q_net, episodes, 200, epsilon)
    
    print(len(tmpSample)) # the total number time steps for 10 episodes
    # save the best model that generates the longest steps over 10 episodes
    if len(tmpSample) >= shortest_len:
        print("best model!save it!")
        torch.save(q_net.state_dict(), "bestmodel_cartpole" + ".pth")
        shortest_len = len(tmpSample)
        
    # update dataset train_samples by adding tempSample
    # dataset contains the latest samples not exceeding 4000
    train_samples += tmpSample
    if len(train_samples) > 4000:
        train_samples = train_samples[len(tmpSample):len(train_samples)]
    
    trainset = RLDataset(train_samples)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=0,pin_memory=True)
    
    TrainNet(target_net, q_net, trainloader, criterion, optimizer, device, epochs, 0.9)

env.close()

dataset: 0 epsilon 1.0
169
dataset: 1 epsilon 1.0
195
dataset: 2 epsilon 0.9932885906040269
247
dataset: 3 epsilon 0.9865771812080537
159
dataset: 4 epsilon 0.9798657718120806
228
dataset: 5 epsilon 0.9731543624161074
228
dataset: 6 epsilon 0.9664429530201343
171
dataset: 7 epsilon 0.9597315436241611
175
dataset: 8 epsilon 0.953020134228188
314
dataset: 9 epsilon 0.9463087248322148
169
dataset: 10 epsilon 0.9395973154362417
220
dataset: 11 epsilon 0.9328859060402686
195
dataset: 12 epsilon 0.9261744966442954
215
dataset: 13 epsilon 0.9194630872483223
228
dataset: 14 epsilon 0.9127516778523491
225
dataset: 15 epsilon 0.906040268456376
297
dataset: 16 epsilon 0.8993288590604028
193
dataset: 17 epsilon 0.8926174496644297
176
dataset: 18 epsilon 0.8859060402684565
184
dataset: 19 epsilon 0.8791946308724834
420
dataset: 20 epsilon 0.8724832214765103
275
dataset: 21 epsilon 0.8657718120805371
191
dataset: 22 epsilon 0.859060402684564
184
dataset: 23 epsilon 0.8523489932885908
194
dataset: 24

Episode finished after 197 timesteps
Episode finished after 200 timesteps
1859
dataset: 139 epsilon 0.07382550335570592
190
dataset: 140 epsilon 0.06711409395973277
1196
dataset: 141 epsilon 0.060402684563759614
195
dataset: 142 epsilon 0.05369127516778646
Episode finished after 198 timesteps
Episode finished after 200 timesteps
Episode finished after 197 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
1855
dataset: 143 epsilon 0.04697986577181331
664
dataset: 144 epsilon 0.04026845637584016
1100
dataset: 145 epsilon 0.03355704697986701
439
dataset: 146 epsilon 0.026845637583893852
336
dataset: 147 epsilon 0.020134228187920697
904
dataset: 148 epsilon 0.013422818791947542
426
dataset: 149 epsilon 0.006711409395974388
914
dataset: 150 epsilon 0.006711409395974388
110
dataset: 151 epsilon 0.006711409395974388
1200
dataset: 152 epsilon 0.006711

Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
1990
best model!save it!
dataset: 229 epsilon 0.006711409395974388
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
1990
best model!save it!
dataset: 230 epsilon 0.006711409395974388
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finishe

dataset: 260 epsilon 0.006711409395974388
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
1990
best model!save it!
dataset: 261 epsilon 0.006711409395974388
639
dataset: 262 epsilon 0.006711409395974388
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
1990
best model!save it!
dataset: 263 epsilon 0.006711409395974388
780
dataset: 264 epsilon 0.00671140939

Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
1990
best model!save it!
dataset: 289 epsilon 0.006711409395974388
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
1990
best model!save it!
dataset: 290 epsilon 0.006711409395974388
1132
dataset: 291 epsilon 0.006711409395974388
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episode finished after 200 timesteps
Episo

In [ ]:
import numpy as np
np.shape(train_samples)

In [ ]:
plt.plot(avg_len, label="average rewards")
plt.legend(loc=4)
plt.show()

In [ ]:
len(avg_len)

In [9]:

test_model = Q_net()
test_model.load_state_dict(torch.load("bestmodel_cartpole.pth"))
test_model.eval()
total_reward = 0.0
with torch.no_grad():
    for i in range (100): 
        done = False
        observation = env.reset()
        t=0
        while not done:
            t += 1
            if i%10 == 0:
                env.render()
            inputT = torch.tensor(observation).float()
            action = torch.argmax(test_model(inputT)).item()
            observation_new, reward, done, info = env.step(action)
            observation = observation_new
            total_reward += reward
            if done:
                if t <= 199:
                    print("Not success at time step", t, "episode", i)

                break
env.close()           


print(total_reward)

20000.0
